In [0]:
%pip install -U databricks-sdk "psycopg[binary]"

In [0]:
dbutils.library.restartPython()

In [0]:
# Seeds the Lakebase operational database with the Olist dataset.
# Re-running this notebook drops and recreates every table (full reset).

CATALOG   = "de_databricks_bp_wave_5"
VOLUME    = f"/Volumes/{CATALOG}/olist_dev_landing/olist_raw"
ENDPOINT  = "projects/olist-capstone/branches/production/endpoints/primary"
HOST      = "ep-noisy-grass-d1gvpml8.database.us-west-2.cloud.databricks.com"
DB        = "databricks_postgres"
PG_SCHEMA = "olist"
USER      = spark.sql("select current_user()").first()[0]

import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def pg_token() -> str:
    """Short-lived OAuth token (1h) used as the Postgres password."""
    return w.postgres.generate_database_credential(endpoint=ENDPOINT).token

def pg_connect():
    return psycopg.connect(host=HOST, port=5432, dbname=DB, user=USER,
                           password=pg_token(), sslmode="require")

print(f"Target: {HOST}/{DB}, schema '{PG_SCHEMA}', user {USER}")

In [0]:
# Operational (OLTP) schema. Column names keep the Olist source spelling,
# including its typos (e.g. product_name_lenght); Silver fixes them later.
# Every table gets updated_at, maintained by a trigger, so downstream
# incremental loads can pick up only rows that changed.

TABLES = {
    "customers": """
        customer_id              text primary key,
        customer_unique_id       text not null,
        customer_zip_code_prefix text,
        customer_city            text,
        customer_state           text""",
    "sellers": """
        seller_id                text primary key,
        seller_zip_code_prefix   text,
        seller_city              text,
        seller_state             text""",
    "products": """
        product_id                 text primary key,
        product_category_name      text,
        product_name_lenght        int,
        product_description_lenght int,
        product_photos_qty         int,
        product_weight_g           int,
        product_length_cm          int,
        product_height_cm          int,
        product_width_cm           int""",
    "orders": """
        order_id                      text primary key,
        customer_id                   text not null,
        order_status                  text not null,
        order_purchase_timestamp      timestamp not null,
        order_approved_at             timestamp,
        order_delivered_carrier_date  timestamp,
        order_delivered_customer_date timestamp,
        order_estimated_delivery_date timestamp""",
    "order_items": """
        order_id            text not null,
        order_item_id       int  not null,
        product_id          text not null,
        seller_id           text not null,
        shipping_limit_date timestamp,
        price               numeric(10,2),
        freight_value       numeric(10,2),
        primary key (order_id, order_item_id)""",
    "order_payments": """
        order_id             text not null,
        payment_sequential   int  not null,
        payment_type         text,
        payment_installments int,
        payment_value        numeric(10,2),
        primary key (order_id, payment_sequential)""",
    "order_reviews": """
        review_id               text not null,
        order_id                text not null,
        review_score            int,
        review_comment_title    text,
        review_comment_message  text,
        review_creation_date    timestamp,
        review_answer_timestamp timestamp,
        primary key (review_id, order_id)""",
}

with pg_connect() as conn, conn.cursor() as cur:
    cur.execute(f"create schema if not exists {PG_SCHEMA}")
    cur.execute(f"drop table if exists {PG_SCHEMA}.smoke_test")
    cur.execute(f"""
        create or replace function {PG_SCHEMA}.set_updated_at() returns trigger
        language plpgsql as $$
        begin
            new.updated_at := now();
            return new;
        end;
        $$""")
    for table, columns in TABLES.items():
        cur.execute(f"drop table if exists {PG_SCHEMA}.{table} cascade")
        cur.execute(f"""
            create table {PG_SCHEMA}.{table} (
                {columns},
                updated_at timestamptz not null default now()
            )""")
        cur.execute(f"""
            create trigger trg_{table}_updated_at
            before update on {PG_SCHEMA}.{table}
            for each row execute function {PG_SCHEMA}.set_updated_at()""")
        print(f"created {PG_SCHEMA}.{table}")
    conn.commit()

In [0]:
# Explicit schemas instead of inferSchema: zip code prefixes must stay
# strings (inference turns '01310' into 1310 and drops the leading zero).

SOURCES = {
    "customers": ("olist_customers_dataset.csv",
        "customer_id string, customer_unique_id string, "
        "customer_zip_code_prefix string, customer_city string, customer_state string"),
    "sellers": ("olist_sellers_dataset.csv",
        "seller_id string, seller_zip_code_prefix string, "
        "seller_city string, seller_state string"),
    "products": ("olist_products_dataset.csv",
        "product_id string, product_category_name string, "
        "product_name_lenght int, product_description_lenght int, product_photos_qty int, "
        "product_weight_g int, product_length_cm int, product_height_cm int, product_width_cm int"),
    "orders": ("olist_orders_dataset.csv",
        "order_id string, customer_id string, order_status string, "
        "order_purchase_timestamp timestamp, order_approved_at timestamp, "
        "order_delivered_carrier_date timestamp, order_delivered_customer_date timestamp, "
        "order_estimated_delivery_date timestamp"),
    "order_items": ("olist_order_items_dataset.csv",
        "order_id string, order_item_id int, product_id string, seller_id string, "
        "shipping_limit_date timestamp, price decimal(10,2), freight_value decimal(10,2)"),
    "order_payments": ("olist_order_payments_dataset.csv",
        "order_id string, payment_sequential int, payment_type string, "
        "payment_installments int, payment_value decimal(10,2)"),
    "order_reviews": ("olist_order_reviews_dataset.csv",
        "review_id string, order_id string, review_score int, "
        "review_comment_title string, review_comment_message string, "
        "review_creation_date timestamp, review_answer_timestamp timestamp"),
}

frames = {}
for table, (file_name, schema) in SOURCES.items():
    frames[table] = (spark.read
        .option("header", True)
        .option("multiLine", True)          # review comments contain line breaks
        .option("escape", '"')
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .option("mode", "FAILFAST")         # stop on malformed rows instead of nulling them
        .schema(schema)
        .csv(f"{VOLUME}/{file_name}"))
    print(f"{table:16s} {frames[table].count():>9,} rows read")

In [0]:
# Bulk-load into the pre-created tables. Append mode keeps our DDL
# (primary keys, defaults, triggers); updated_at is filled by its default.
# Serverless compute only allows writes through the native "postgresql"
# connector; the generic "jdbc" format is read-only there.

token = pg_token()

for table, df in frames.items():
    (df.write.format("postgresql")
        .option("host", HOST)
        .option("port", "5432")
        .option("database", DB)
        .option("dbtable", f"{PG_SCHEMA}.{table}")
        .option("user", USER)
        .option("password", token)
        .option("batchsize", 5000)
        .mode("append")
        .save())
    print(f"loaded {PG_SCHEMA}.{table}")